# Campaign outcomes per trained model version

Every campaign in `D:/twdata/runs/human` is assigned to the model version live when it
started: the latest retrain (from the `session_*.json` reports) before its first decision
timestamp. `trained_at`/`rows` identify the version; both are null for campaigns played
before any model was fitted.

In [1]:
import glob, json, os, sqlite3, time
import pandas as pd

RUNS = r"D:\twdata\runs\human"

retrains = []
for rp in sorted(glob.glob(os.path.join(RUNS, "session_*.json"))):
    rep = json.load(open(rp, encoding="utf-8"))
    for c in rep.get("campaigns") or []:
        r = c.get("retrain")
        if r and r.get("trained"):
            retrains.append({"ts": c["started"], "rows": r.get("rows")})
retrains.sort(key=lambda r: r["ts"])

def version_of(ts):
    ver = (float("nan"), float("nan"))
    for r in retrains:
        if ts >= r["ts"]:
            ver = (r["ts"], r["rows"])
    return ver

camps = {}
for db in sorted(glob.glob(os.path.join(RUNS, "*", "decisions.sqlite"))):
    con = sqlite3.connect("file:%s?mode=ro" % db.replace("\\", "/"), uri=True)
    try:
        for cid, t0 in con.execute(
                "SELECT campaign_id, MIN(ts) FROM decision_points GROUP BY campaign_id"):
            camps.setdefault(cid, {}).update(start_ts=t0)
        for cid, n, setts, lvl in con.execute(
                "SELECT campaign_id, COUNT(*), MAX(settlements), MAX(lord_level)"
                " FROM target_rows GROUP BY campaign_id"):
            camps.setdefault(cid, {}).update(turns_played=n, max_settlements=setts,
                                             max_lord_level=lvl)
    finally:
        con.close()

per_camp = pd.DataFrame([dict(campaign=cid, trained_ts=version_of(c["start_ts"])[0],
                              rows=version_of(c["start_ts"])[1], **c)
                         for cid, c in camps.items() if c.get("start_ts")])
df = (per_camp.groupby(["trained_ts", "rows"], dropna=False)
      .agg(campaigns=("campaign", "count"),
           avg_turns_played=("turns_played", "mean"),
           avg_max_settlements=("max_settlements", "mean"),
           best_settlements=("max_settlements", "max"),
           avg_max_lord_level=("max_lord_level", "mean"),
           best_lord_level=("max_lord_level", "max"))
      .round(2)
      .reset_index()
      .sort_values("trained_ts", na_position="first")
      .reset_index(drop=True))
df.insert(0, "trained_at", df.pop("trained_ts").map(
    lambda t: time.strftime("%Y-%m-%d %H:%M", time.localtime(t)) if t == t else None))

In [2]:
df['settlement_expansion_rate'] = df['avg_max_settlements'] / (df['avg_turns_played'] - 1).clip(lower=1)
df['Legendary_lord_level_rate'] = df['avg_max_lord_level']  / (df['avg_turns_played'] - 1).clip(lower=1)
df

,trained_at,rows,campaigns,avg_turns_played,avg_max_settlements,best_settlements,avg_max_lord_level,best_lord_level,settlement_expansion_rate,Legendary_lord_level_rate
0,NaN,NaN,10,5.60,1.20,2.0,2.00,4.0,0.260870,0.434783
1,2026-08-03 18:00,438.0,4,9.75,1.25,2.0,1.75,2.0,0.142857,0.200000
2,2026-08-03 19:23,778.0,10,4.67,1.67,3.0,2.89,5.0,0.455041,0.787466
3,2026-08-03 21:09,1156.0,3,2.67,1.00,1.0,1.67,3.0,0.598802,1.000000
4,2026-08-03 21:37,1237.0,2,3.00,0.50,1.0,2.00,3.0,0.250000,1.000000
5,2026-08-03 22:00,1287.0,10,5.60,0.90,2.0,2.50,5.0,0.195652,0.543478
6,2026-08-03 23:59,1714.0,3,4.33,0.67,1.0,2.00,3.0,0.201201,0.600601
7,2026-08-04 00:30,1803.0,12,5.56,1.89,4.0,3.11,6.0,0.414474,0.682018
8,2026-08-04 05:18,2266.0,10,8.20,1.90,4.0,4.10,7.0,0.263889,0.569444
9,2026-08-04 08:15,3011.0,10,6.80,1.10,3.0,3.10,7.0,0.189655,0.534483
